# PDF → Audiobook with Kokoro TTS

**Voice:** `am_fenrir` (American male)  
**Runtime:** Set to GPU in Runtime → Change runtime type → T4 GPU

Steps:
1. Install dependencies
2. Upload your PDF
3. Extract & clean text
4. Convert to audio
5. Download the result

## Step 1: Install Dependencies

In [ ]:
!pip install -q "kokoro>=0.9.2" soundfile pdfminer.six
!apt-get -qq -y install espeak-ng > /dev/null 2>&1
print('Done')

## Step 2: Upload PDF

In [ ]:
from google.colab import files

uploaded = files.upload()  # drag & drop your PDF here
pdf_path = list(uploaded.keys())[0]
print(f'Uploaded: {pdf_path}')

## Step 3: Extract & Clean Text

In [ ]:
from pdfminer.high_level import extract_text
import re

# --- Config ---
VOICE      = 'am_fenrir'   # change to any Kokoro voice
LANG_CODE  = 'a'           # 'a' = American English, 'b' = British English

# --- Extract ---
raw_text = extract_text(pdf_path)
raw_lines = raw_text.split('\n')
print(f'Extracted {len(raw_lines)} lines, {len(raw_text)} characters')

# --- Tag lines ---
SECTION_TITLE = 'section'
SUBSECTION    = 'subsection'
BLANK         = 'blank'
BODY          = 'body'
SKIP          = 'skip'

# Running headers to strip — add any repeated page headers from your book here
SKIP_LINES = {'THE LOGIC OF BUSINESS STRATEGY', 'STRATEGIC CONCEPTS'}

tagged = []
for line in raw_lines:
    s = line.strip()
    if re.match(r'^\d+$', s):
        tagged.append((SKIP, ''))
    elif re.match(r'^[^a-zA-Z\s]*$', s) and s:
        tagged.append((SKIP, ''))
    elif s in SKIP_LINES:
        tagged.append((SKIP, ''))
    elif not s:
        tagged.append((BLANK, ''))
    elif re.match(r'^\d+\.\s+[A-Z][A-Z\s]+$', s):
        tagged.append((SECTION_TITLE, s))
    elif (2 <= len(s.split()) <= 7
          and not s.endswith(('.', ',', ';', ':'))
          and re.match(r'^[A-Z][a-z]+(\s+(of|the|and|in|a|an|to|for|with|by|on|at|from)|(\s+[A-Z][a-z]+))+$', s)):
        tagged.append((SUBSECTION, s))
    else:
        tagged.append((BODY, s))

# --- Fix hyphenated line breaks ---
merged = []
i = 0
while i < len(tagged):
    tag, content = tagged[i]
    if tag == BODY and content.endswith('-') and i + 1 < len(tagged):
        j = i + 1
        while j < len(tagged) and tagged[j][0] in (BLANK, SKIP):
            j += 1
        if j < len(tagged) and tagged[j][0] == BODY:
            merged.append((BODY, content[:-1] + tagged[j][1]))
            i = j + 1
            continue
    merged.append((tag, content))
    i += 1

# --- Group into paragraphs ---
paragraphs = []
buf = []
for tag, content in merged:
    if tag == SKIP:
        continue
    elif tag in (SECTION_TITLE, SUBSECTION):
        if buf:
            paragraphs.append((BODY, ' '.join(buf)))
            buf = []
        paragraphs.append((tag, content))
    elif tag == BLANK:
        if buf:
            paragraphs.append((BODY, ' '.join(buf)))
            buf = []
        paragraphs.append((BLANK, ''))
    else:
        if content:
            buf.append(content)
if buf:
    paragraphs.append((BODY, ' '.join(buf)))

# --- Merge short orphan fragments ---
merged2 = []
i = 0
while i < len(paragraphs):
    tag, content = paragraphs[i]
    if (tag == BODY and len(content.split()) < 8
            and not content.endswith(('.', '?', '!', '"', '\u201d'))
            and i + 1 < len(paragraphs)):
        j = i + 1
        while j < len(paragraphs) and paragraphs[j][0] == BLANK:
            j += 1
        if j < len(paragraphs) and paragraphs[j][0] == BODY:
            paragraphs[j] = (BODY, content + ' ' + paragraphs[j][1])
            i += 1
            continue
    merged2.append((tag, content))
    i += 1
paragraphs = merged2

# --- Collapse consecutive blanks ---
deduped = []
for item in paragraphs:
    if item[0] == BLANK and deduped and deduped[-1][0] == BLANK:
        continue
    deduped.append(item)

# --- Build final text chunks for TTS (split at section/subsection boundaries) ---
chunks = []
current = []
for tag, content in deduped:
    if tag in (SECTION_TITLE, SUBSECTION) and current:
        chunks.append(' '.join(current))
        current = [content + '.']
    elif tag == BLANK:
        current.append('')  # paragraph break — Kokoro handles natural pausing
    elif tag in (SECTION_TITLE, SUBSECTION):
        current.append(content + '.')
    else:
        current.append(content)
if current:
    chunks.append(' '.join(current))

print(f'Text split into {len(chunks)} chunks for TTS')
print('\n--- Preview of first chunk ---')
print(chunks[0][:500])

## Step 4: Convert to Audio

In [ ]:
from kokoro import KPipeline
import soundfile as sf
import numpy as np
import torch

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using device: {device}')

pipeline = KPipeline(lang_code=LANG_CODE)

all_audio = []
silence = np.zeros(int(24000 * 0.6))  # 600ms silence between chunks

for i, chunk in enumerate(chunks):
    if not chunk.strip():
        continue
    print(f'Chunk {i+1}/{len(chunks)}...', end=' ')
    chunk_audio = [audio for _, _, audio in pipeline(chunk, voice=VOICE)]
    if chunk_audio:
        all_audio.append(np.concatenate(chunk_audio))
        all_audio.append(silence)
    print('done')

final_audio = np.concatenate(all_audio)
output_file = pdf_path.replace('.pdf', f'_{VOICE}.wav')
sf.write(output_file, final_audio, 24000)

duration_mins = len(final_audio) / 24000 / 60
print(f'\nSaved: {output_file}')
print(f'Duration: {duration_mins:.1f} minutes')

## Step 5: Download

In [ ]:
from google.colab import files
files.download(output_file)